In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Layer 1: Extreme ESI 1 Binary Feature Engineered Logistic Regressor (`models/lr_feng_esi1_extreme.ipynb`)

This notebook trains a **Binary Logistic Regressor** to classify **ESI 1 (Highest Acuity) vs. Not ESI 1** and includes **Overfitting / Underfitting Diagnostic Plots**:
- **Binary Target Output**: Predicts whether a patient is **ESI 1** (`"1"`) or **Not ESI 1** (`"not_1"`).
- **Feature Set (13 Predictors)**: `age`, `gender`, `cc_breathingdifficulty`, and the **10 Clinical Feature Engineering flags** defined in `TODO.md` (`is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`).
- **Configurable Target Class Subsampling**:
  - `keep_ratio_1 = 1.00` (Keeps 100% of ESI 1 rows).
  - `keep_ratio_not_1 = 0.10` (Keeps 10% of 'not_1' rows, reducing 'not_1' by 90%).
- **Overfit / Underfit Diagnostics**:
  - **Train vs. Validation vs. Test Metrics Comparison Bar Chart** (Accuracy, ROC-AUC, Log Loss).
  - **Train vs. Validation vs. Test Overlay ROC Curves**.
  - **Learning Curve Plot** (Performance vs. Training Sample Size).

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(nnet)
library(dplyr)
library(ggplot2)
library(pROC)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Compute 10 FE Flags + Demographics & Apply Binary Class Subsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0

# Compute 10 Clinical Feature Engineering flags + Age + Gender + cc_breathingdifficulty
df_feng <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)

df_feng[[target_col]] <- raw_df[[target_col]]

# Create Binary ESI 1 Target: '1' vs 'not_1'
raw_esi <- as.character(df_feng[[target_col]])
df_feng$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"),
                                levels = c("1", "not_1"))

# ---------------------------------------------------------
# Subsample Binary Target Classes (Configurable keep ratios per class)
# ---------------------------------------------------------
keep_ratio_1     <- 1.00  # Keep 100% of ESI 1 rows (configurable)
keep_ratio_not_1 <- 0.10  # Keep 10% of 'not_1' rows (reduces 'not_1' by 90%)

idx_1     <- which(df_feng$target_layer1 == "1")
idx_not_1 <- which(df_feng$target_layer1 == "not_1")

kept_1     <- sample(idx_1,     size = round(length(idx_1)     * keep_ratio_1))
kept_not_1 <- sample(idx_not_1, size = round(length(idx_not_1) * keep_ratio_not_1))

df_feng <- df_feng[sort(c(kept_1, kept_not_1)), ]

cat(sprintf("Binary ESI 1 FE Dataset Ready (Class Ratios: ESI 1=%.0f%%, Not ESI 1=%.0f%%): %d rows x %d cols\n",
            keep_ratio_1 * 100, keep_ratio_not_1 * 100, nrow(df_feng), ncol(df_feng)))
cat("Feature Engineered Input Columns (13):\n", paste(setdiff(names(df_feng), c(target_col, "target_layer1")), collapse = ", "), "\n")
cat("Binary Target Distribution:\n")
print(table(df_feng$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Continuous Feature Scaling
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

# Stratified Test split (15%)
in_train_val <- createDataPartition(df_feng$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_feng[in_train_val, ]
test_df      <- df_feng[-in_train_val, ]

# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Standardize continuous feature (age strictly)
cont_cols <- "age"
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))

train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)

cat(sprintf("Partition sizes:\n  Train: %d rows\n  Val:   %d rows\n  Test:  %d rows\n",
            nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Binary ESI 1 Feature Engineered Logistic Regressor
# ---------------------------------------------------------
set.seed(config$training$random_state)

feat_names <- setdiff(names(train_df), c(target_col, "target_layer1"))
formula_lr <- as.formula(paste("target_layer1 ~", paste(feat_names, collapse = " + ")))

cat("Training Binary ESI 1 Feature Engineered Logistic Regressor...\n")
lr_esi1_model <- multinom(formula_lr, data = train_df, trace = FALSE, MaxNWts = 5000)

cat("Binary ESI 1 Logistic Regression training complete!\n")
print(summary(lr_esi1_model))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive Evaluation Across Train, Validation, and Test Sets
# ---------------------------------------------------------
calc_log_loss <- function(actual_factor, prob_matrix, eps = 1e-15) {
  if (!is.matrix(prob_matrix)) {
    prob_matrix <- cbind(1 - prob_matrix, prob_matrix)
    colnames(prob_matrix) <- levels(actual_factor)
  }
  prob_matrix <- pmax(pmin(prob_matrix, 1 - eps), eps)
  prob_matrix <- prob_matrix / rowSums(prob_matrix)
  classes <- colnames(prob_matrix)
  N <- length(actual_factor)
  
  log_probs <- numeric(N)
  for (i in 1:N) {
    act_cls <- as.character(actual_factor[i])
    if (act_cls %in% classes) {
      log_probs[i] <- log(prob_matrix[i, act_cls])
    } else {
      log_probs[i] <- log(eps)
    }
  }
  return(-mean(log_probs))
}

evaluate_esi1_lr <- function(model, data, set_name) {
  prob_res <- predict(model, newdata = data, type = "probs")
  target_classes <- levels(data$target_layer1)
  
  if (is.matrix(prob_res)) {
    prob_matrix <- prob_res
  } else {
    prob_matrix <- cbind(1 - prob_res, prob_res)
    colnames(prob_matrix) <- c("1", "not_1")
  }
  
  max_idx <- max.col(prob_matrix, ties.method = "first")
  pred_factor <- factor(colnames(prob_matrix)[max_idx], levels = target_classes)
  actual_factor <- factor(data$target_layer1, levels = target_classes)
  
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  roc_auc <- tryCatch({
    as.numeric(pROC::roc(actual_factor == "1", prob_matrix[, "1"], quiet = TRUE)$auc)
  }, error = function(e) NA)
  
  log_loss <- calc_log_loss(actual_factor, prob_matrix)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   BINARY ESI 1 FEATURE ENGINEERED LR - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Accuracy   : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  ROC-AUC    : %.4f\n", roc_auc))
  cat(sprintf("  Log Loss   : %.4f\n", log_loss))
  cat("\nTarget Class Counts (Actual vs Predicted Comparison):\n")
  class_counts_df <- data.frame(
    Class = target_classes,
    Actual_Count = as.numeric(table(actual_factor)[target_classes]),
    Predicted_Count = as.numeric(table(pred_factor)[target_classes]),
    Diff = as.numeric(table(pred_factor)[target_classes]) - as.numeric(table(actual_factor)[target_classes])
  )
  print(class_counts_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, roc_auc = roc_auc, log_loss = log_loss, prob_matrix = prob_matrix, actual_factor = actual_factor))
}

# Evaluate on Train, Validation, and Test Sets
res_train <- evaluate_esi1_lr(lr_esi1_model, train_df, "Train")
res_val   <- evaluate_esi1_lr(lr_esi1_model, val_df,   "Validation")
res_test  <- evaluate_esi1_lr(lr_esi1_model, test_df,  "Test")

# Compare Metrics Summary Across Splits
metrics_summary <- data.frame(
  Split    = c("Train", "Validation", "Test"),
  Accuracy = c(res_train$acc, res_val$acc, res_test$acc),
  ROC_AUC  = c(res_train$roc_auc, res_val$roc_auc, res_test$roc_auc),
  Log_Loss = c(res_train$log_loss, res_val$log_loss, res_test$log_loss)
)

cat("=== OVERALL METRICS COMPARISON (TRAIN vs VALIDATION vs TEST) ===\n")
print(metrics_summary)

# Diagnose Overfitting / Underfitting
delta_acc <- res_train$acc - res_val$acc
delta_auc <- res_train$roc_auc - res_val$roc_auc

cat("\n=== DIAGNOSTIC EVALUATION SUMMARY ===\n")
cat(sprintf("  - Accuracy Delta (Train - Val):  %+.4f\n", delta_acc))
cat(sprintf("  - ROC-AUC Delta (Train - Val):   %+.4f\n", delta_auc))
if (delta_auc > 0.05) {
  cat("  -> DIAGNOSIS: Potential OVERFITTING detected (Train ROC-AUC is significantly higher than Validation).\n")
} else if (res_val$acc < 0.60 && res_train$acc < 0.60) {
  cat("  -> DIAGNOSIS: Potential UNDERFITTING detected (Low accuracy on both Train & Validation).\n")
} else {
  cat("  -> DIAGNOSIS: WELL-GENERALIZED MODEL (Train, Validation, and Test metrics are closely aligned).\n")
}

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Overfitting / Underfitting Diagnostic Visualizations
# ---------------------------------------------------------
# Plot 1: Train vs Validation vs Test Metrics Comparison Bar Chart
library(tidyr)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "ROC_AUC", "Log_Loss"), names_to = "Metric", values_to = "Score")

p1 <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3.5) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics Comparison",
       subtitle = "Close agreement between Train and Val indicates good generalization without overfitting",
       y = "Metric Value", x = "") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "top")

print(p1)

# Plot 2: Overlaid ROC Curves (Train vs Validation vs Test)
roc_tr <- pROC::roc(res_train$actual_factor == "1", res_train$prob_matrix[, "1"], quiet = TRUE)
roc_va <- pROC::roc(res_val$actual_factor == "1",   res_val$prob_matrix[, "1"], quiet = TRUE)
roc_te <- pROC::roc(res_test$actual_factor == "1",  res_test$prob_matrix[, "1"], quiet = TRUE)

df_roc_tr <- data.frame(FPR = 1 - roc_tr$specificities, TPR = roc_tr$sensitivities, Split = sprintf("Train (AUC = %.3f)", roc_tr$auc))
df_roc_va <- data.frame(FPR = 1 - roc_va$specificities, TPR = roc_va$sensitivities, Split = sprintf("Validation (AUC = %.3f)", roc_va$auc))
df_roc_te <- data.frame(FPR = 1 - roc_te$specificities, TPR = roc_te$sensitivities, Split = sprintf("Test (AUC = %.3f)", roc_te$auc))

df_roc_all <- rbind(df_roc_tr, df_roc_va, df_roc_te)

p2 <- ggplot(df_roc_all, aes(x = FPR, y = TPR, color = Split)) +
  geom_line(size = 1.2) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "gray50") +
  theme_minimal() +
  scale_color_manual(values = c("#2b5c8f", "#e07a5f", "#81b29a")) +
  labs(title = "ROC Curves Comparison Across Splits",
       subtitle = "Overlaid ROC curves to diagnose overfitting / underfitting",
       x = "False Positive Rate (1 - Specificity)", y = "True Positive Rate (Sensitivity)") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "bottom")

print(p2)

# Plot 3: Learning Curve (Training Sample Fraction vs Performance)
sample_fractions <- c(0.2, 0.4, 0.6, 0.8, 1.0)
lc_results <- data.frame()

set.seed(config$training$random_state)
for (frac in sample_fractions) {
  n_sub <- round(nrow(train_df) * frac)
  sub_idx <- sample(1:nrow(train_df), size = n_sub)
  sub_train <- train_df[sub_idx, ]
  
  fit_sub <- multinom(formula_lr, data = sub_train, trace = FALSE, MaxNWts = 5000)
  
  tr_prob <- predict(fit_sub, newdata = sub_train, type = "probs")
  va_prob <- predict(fit_sub, newdata = val_df,   type = "probs")
  
  tr_auc <- as.numeric(pROC::roc(sub_train$target_layer1 == "1", if(is.matrix(tr_prob)) tr_prob[,"1"] else tr_prob, quiet = TRUE)$auc)
  va_auc <- as.numeric(pROC::roc(val_df$target_layer1 == "1",   if(is.matrix(va_prob)) va_prob[,"1"] else va_prob, quiet = TRUE)$auc)
  
  lc_results <- rbind(lc_results, data.frame(Fraction = frac * 100, Train_AUC = tr_auc, Val_AUC = va_auc))
}

lc_long <- lc_results %>%
  pivot_longer(cols = c("Train_AUC", "Val_AUC"), names_to = "Split", values_to = "ROC_AUC") %>%
  mutate(Split = ifelse(Split == "Train_AUC", "Train", "Validation"))

p3 <- ggplot(lc_long, aes(x = Fraction, y = ROC_AUC, color = Split, group = Split)) +
  geom_line(size = 1.2) +
  geom_point(size = 3) +
  theme_minimal() +
  scale_color_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f")) +
  labs(title = "Learning Curve: Sample Size vs ROC-AUC",
       subtitle = "Convergence of Train and Validation curves shows sample size adequacy",
       x = "Training Data Percentage (%)", y = "ROC-AUC Score") +
  theme(plot.title = element_text(face = "bold", size = 14),
        legend.position = "top")

print(p3)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Binary ESI 1 Model Artifacts
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

model_path <- file.path(deploy_dir, "lr_feng_esi1_extreme_model.rds")
saveRDS(list(model = lr_esi1_model, preproc = preproc), file = model_path)
cat("Binary ESI 1 Feature Engineered Logistic Regressor model saved to:", model_path, "\n")